# Remove noise from BTC prices and pass it in a DNN-HMM

We will use the Binance API to get BTC prices. We will use the following methods to de noise our data:
1) Exponential Moving Average (EMA)
2) Wavelet transformation (db4, MODWT)
3) Singular Spectrum Analysis (SSA)

We will evaluate multiple DNNHMM models for different wavelets and for different SSA, EMA and RSI periods. Our DNNHMM will only receive the RSI values as input for regime detection.

We will also evaluate our models for smaller time frames.

https://gullayeshwantkumarruler.medium.com/wavelet-transform-a-better-alternative-to-ema-for-trend-analysis-760541ac3d7c

https://www.sciencedirect.com/science/article/abs/pii/S0261560621001832

https://www.sciencedirect.com/science/article/pii/S1018363918307554

In [4]:
import requests
import pandas as pd
import time
from datetime import datetime
import math

def get_btc_close_prices(start_date, end_date, interval="1d"):
    """
    Download BTC/USDT historical close prices from Binance API with customizable frequency
    
    Args:
        start_date (str): Start date in format 'YYYY-MM-DD'
        end_date (str): End date in format 'YYYY-MM-DD'
        interval (str, optional): Data frequency. Valid options:
            - Minutes: '1m', '3m', '5m', '15m', '30m'
            - Hours: '1h', '2h', '4h', '6h', '8h', '12h'
            - Days: '1d', '3d'
            - Weeks: '1w'
            - Months: '1M'
            Default is '1d' (daily data)
        
    Returns:
        pandas.Series: Series containing the historical close prices
    """
    # Validate interval parameter
    valid_intervals = ['1m', '3m', '5m', '15m', '30m', '1h', '2h', '4h', '6h', '8h', '12h', '1d', '3d', '1w', '1M']
    if interval not in valid_intervals:
        raise ValueError(f"Invalid interval: {interval}. Valid options are: {', '.join(valid_intervals)}")
    
    # Determine appropriate time window based on interval
    # For higher frequency data (especially minutes), we may need to split requests into smaller time chunks
    time_chunk_days = 1000  # Default chunk size in days (for daily data)
    
    if interval.endswith('m'):
        # For minute data, reduce time chunk size significantly
        # 1000 minutes would be less than a day, so we'll use chunks of ~7 days for minute data
        time_chunk_days = 7
    elif interval.endswith('h'):
        # For hourly data, reduce time chunk size moderately
        # 1000 hours would be ~41 days, so we'll use chunks of ~30 days for hourly data
        time_chunk_days = 30
    
    # Convert dates to milliseconds timestamp
    start_ts = int(datetime.strptime(start_date, '%Y-%m-%d').timestamp() * 1000)
    end_ts = int(datetime.strptime(end_date, '%Y-%m-%d').timestamp() * 1000)
    
    # Binance API endpoint for historical klines (candlestick) data
    base_url = "https://api.binance.com/api/v3/klines"
    
    # Parameters for the API request
    symbol = "BTCUSDT"
    limit = 1000  # Maximum number of data points per request
    
    # Calculate total time range in milliseconds
    total_time_ms = end_ts - start_ts
    
    # Calculate time chunk size in milliseconds
    time_chunk_ms = time_chunk_days * 24 * 60 * 60 * 1000
    
    # Calculate number of iterations needed
    iterations = math.ceil(total_time_ms / time_chunk_ms)
    
    all_data = []
    current_start_ts = start_ts
    
    print(f"Downloading BTC close prices from {start_date} to {end_date} with {interval} interval...")
    
    # Iterate through time chunks to get all data
    for i in range(iterations):
        current_end_ts = min(current_start_ts + time_chunk_ms, end_ts)
        
        # Set parameters for the current request
        params = {
            'symbol': symbol,
            'interval': interval,
            'startTime': current_start_ts,
            'endTime': current_end_ts,
            'limit': limit
        }
        
        # Make the API request
        response = requests.get(base_url, params=params)
        
        # Check if request was successful
        if response.status_code == 200:
            data = response.json()
            
            # Extract only timestamp and close price
            for candle in data:
                all_data.append([candle[0], candle[4]])  # candle[0]=timestamp, candle[4]=close
            
            # Update the start timestamp for the next iteration
            current_start_ts = current_end_ts + 1
            
            # Respect Binance API rate limits
            time.sleep(0.1)
            
            print(f"Progress: {i+1}/{iterations} chunks downloaded - {len(all_data)} data points so far")
        else:
            print(f"Error: {response.status_code}, {response.text}")
            # If we hit an error, wait a bit longer before retrying
            time.sleep(1)
            # Retry the same chunk with a smaller window if possible
            if time_chunk_ms > 24 * 60 * 60 * 1000:  # If chunk is larger than 1 day
                time_chunk_ms = time_chunk_ms // 2
                iterations = math.ceil(total_time_ms / time_chunk_ms)
                print(f"Reducing time chunk size and retrying. New chunk size: {time_chunk_ms/(24*60*60*1000):.1f} days")
                current_start_ts = start_ts  # Reset to beginning
                all_data = []  # Reset collected data
                i = -1  # Reset iteration counter (will be incremented to 0 at start of next loop)
                continue
            else:
                break
    
    if not all_data:
        raise ValueError("No data retrieved from API. Check parameters and try again.")
    
    # Create a DataFrame with only timestamp and close price
    df = pd.DataFrame(all_data, columns=['timestamp', 'close'])
    
    # Convert timestamp to datetime
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    
    # Convert close prices to float
    df['close'] = df['close'].astype(float)
    
    # Set timestamp as index
    df = df.set_index('timestamp')
    
    # Sort index to ensure data is chronological
    df = df.sort_index()
    
    # Return only the close prices as a Series
    return df

# Example usage
# Download BTC close prices from 2018-01-01 to 2024-12-31 (daily by default)
raw_data = get_btc_close_prices('2018-01-01', '2024-12-31')

raw_data.head()

Progress: 1/3 chunks downloaded - 1000 data points so far
Progress: 2/3 chunks downloaded - 2000 data points so far
Progress: 3/3 chunks downloaded - 2556 data points so far


,close
timestamp,
2018-01-01,13380.00
2018-01-02,14675.11
2018-01-03,14919.51
2018-01-04,15059.54
2018-01-05,16960.39


#### EMA + RSI periods
EMA 20 + RSI 14
Why: The 20-day EMA is a popular intermediate smoother—it reduces daily noise while staying responsive to trends. Pairing it with the classic 14-period RSI (Wilder’s default) balances noise reduction with momentum detection.

Use Case: Daily charts for swing trading stocks or forex. It’s a solid middle ground for most markets.

EMA 50 + RSI 20
Why: A 50-day EMA is a strong smoother, often used for longer-term trends, and a 20-period RSI further filters out short-term wiggles. This combo prioritizes clarity over speed.

Use Case: Longer-term investing or trading less volatile assets (e.g., large-cap stocks, indices) on daily or weekly charts.

EMA 10 + RSI 10
Why: The 10-day EMA keeps things fairly responsive while cutting some noise, and a 10-period RSI matches its pace for quicker signals with moderate smoothing.

Use Case: Shorter-term trading (e.g., intraday on 1-hour charts) or more volatile assets like crypto or small-cap stocks.

EMA 30 + RSI 25
Why: A 30-day EMA smooths well without excessive lag, and a 25-period RSI is less common but strikes a nice balance between noise reduction and trend sensitivity.

Use Case: A compromise for swing traders wanting smoother signals than EMA 20/RSI 14 but faster response than EMA 50/RSI 20.

#### Wavelet Transformation
It seems likely that for denoising Bitcoin prices, common mother wavelets include Haar, Daubechies (db2, db4, db6), and Biorthogonal (bior1.3, bior2.2), suitable for capturing price volatility.

#### SSA periods
The evidence leans toward choosing SSA window lengths based on expected cycles, such as 5, 10, 20, 50, 100, and 200 days for daily data, adjusted for data frequency.

In [7]:
import numpy as np
import pywt
from scipy import signal
import matplotlib.pyplot as plt

# Function to calculate RSI
def calculate_rsi(series, period=14):
    """
    Calculate Relative Strength Index (RSI) for a price series.
    
    Args:
        series (pandas.Series): Price series
        period (int): RSI period
        
    Returns:
        pandas.Series: RSI values
    """
    # Calculate price changes
    delta = series.diff()
    
    # Separate gains and losses
    gains = delta.where(delta > 0, 0)
    losses = -delta.where(delta < 0, 0)
    
    # Calculate average gains and losses
    avg_gains = gains.rolling(window=period).mean()
    avg_losses = losses.rolling(window=period).mean()
    
    # Calculate RS (Relative Strength)
    rs = avg_gains / avg_losses
    
    # Calculate RSI
    rsi = 100 - (100 / (1 + rs))
    
    return rsi

# Function to calculate EMA
def calculate_ema(series, period):
    """
    Calculate Exponential Moving Average (EMA) for a price series.
    
    Args:
        series (pandas.Series): Price series
        period (int): EMA period
        
    Returns:
        pandas.Series: EMA values
    """
    return series.ewm(span=period, adjust=False).mean()

# Function for wavelet denoising
def wavelet_denoise(series, wavelet='db4', level=1):
    """
    Apply wavelet denoising to a price series.
    
    Args:
        series (pandas.Series): Price series
        wavelet (str): Wavelet type ('haar', 'db4', etc.)
        level (int): Decomposition level
        
    Returns:
        pandas.Series: Denoised series
    """
    # Convert to numpy array
    data = series.values
    
    # Perform wavelet decomposition
    coeffs = pywt.wavedec(data, wavelet, level=level)
    
    # Apply thresholding to detail coefficients
    for i in range(1, len(coeffs)):
        # Calculate threshold (using VisuShrink/universal threshold)
        threshold = np.sqrt(2 * np.log(len(data))) * np.std(coeffs[i]) / np.sqrt(len(coeffs[i]))
        
        # Apply soft thresholding
        coeffs[i] = pywt.threshold(coeffs[i], threshold, mode='soft')
    
    # Reconstruct signal
    denoised_data = pywt.waverec(coeffs, wavelet)
    
    # Ensure same length as original (reconstruction might alter length slightly)
    denoised_data = denoised_data[:len(data)]
    
    # Convert back to pandas Series with same index
    return pd.Series(denoised_data, index=series.index)

# Function for Singular Spectrum Analysis (SSA) denoising
def ssa_denoise(series, window_length, components=None):
    """
    Apply Singular Spectrum Analysis (SSA) for denoising a price series.
    
    Args:
        series (pandas.Series): Price series
        window_length (int): Window length for SSA
        components (list, optional): List of components to reconstruct signal (e.g., [0, 1])
                                     If None, first half of components are used
        
    Returns:
        pandas.Series: Denoised series
    """
    # Convert to numpy array
    data = series.values
    N = len(data)
    
    # Create trajectory matrix (embedding)
    K = N - window_length + 1
    trajectory_matrix = np.zeros((window_length, K))
    
    for i in range(K):
        trajectory_matrix[:, i] = data[i:i+window_length]
    
    # Singular Value Decomposition
    U, sigma, Vt = np.linalg.svd(trajectory_matrix, full_matrices=False)
    
    # Determine components to use for reconstruction
    if components is None:
        components = list(range(window_length // 2))
    
    # Reconstruct signal using selected components
    reconstructed = np.zeros_like(trajectory_matrix)
    
    for i in components:
        reconstructed += sigma[i] * np.outer(U[:, i], Vt[i, :])
    
    # Diagonal averaging to convert reconstructed matrix back to a time series
    denoised_data = np.zeros(N)
    count = np.zeros(N)
    
    for i in range(window_length):
        for j in range(K):
            denoised_data[i+j] += reconstructed[i, j]
            count[i+j] += 1
    
    denoised_data /= count
    
    # Convert back to pandas Series with same index
    return pd.Series(denoised_data, index=series.index)

# Main execution
if __name__ == "__main__":
    # Download BTC close prices (adjust dates as needed)
    raw_data = get_btc_close_prices('2018-01-01', '2024-12-31')
    
    # Create a dictionary to store all denoised series and original
    price_data = {'original': raw_data['close']}
    
    # Apply EMA denoising with different periods
    ema_periods = [10, 20, 30, 50]
    for period in ema_periods:
        price_data[f'ema_{period}'] = calculate_ema(raw_data['close'], period)
    
    # Apply wavelet denoising with different mother wavelets
    wavelets = ['haar', 'db2', 'db4', 'db6', 'bior1.3', 'bior2.2']
    for wavelet in wavelets:
        price_data[f'wavelet_{wavelet}'] = wavelet_denoise(raw_data['close'], wavelet=wavelet, level=3)
    
    # Apply SSA denoising with different window lengths
    ssa_windows = [5, 10, 20, 50, 100, 200]
    for window in ssa_windows:
        # Only apply SSA if we have enough data points
        if len(raw_data) >= window * 2:
            price_data[f'ssa_{window}'] = ssa_denoise(raw_data['close'], window_length=window)
    
    # Create DataFrame for RSI calculations
    rsi_data = pd.DataFrame(index=raw_data.index)
    
    # Calculate RSI for original data and each denoised series with different periods
    rsi_periods = [10, 14, 20, 25]
    
    # For EMA denoised data (specified combinations)
    ema_rsi_combos = {
        'ema_20': 14,  # EMA 20 + RSI 14
        'ema_50': 20,  # EMA 50 + RSI 20
        'ema_10': 10,  # EMA 10 + RSI 10
        'ema_30': 25   # EMA 30 + RSI 25
    }
    
    for ema_key, rsi_period in ema_rsi_combos.items():
        col_name = f"{ema_key}_rsi_{rsi_period}"
        rsi_data[col_name] = calculate_rsi(price_data[ema_key], period=rsi_period)
    
    # For wavelet denoised data
    for wavelet in wavelets:
        for period in rsi_periods:
            col_name = f"wavelet_{wavelet}_rsi_{period}"
            rsi_data[col_name] = calculate_rsi(price_data[f'wavelet_{wavelet}'], period=period)
    
    # For SSA denoised data
    for window in ssa_windows:
        # Only process if the SSA was calculated (enough data points)
        if f'ssa_{window}' in price_data:
            for period in rsi_periods:
                col_name = f"ssa_{window}_rsi_{period}"
                rsi_data[col_name] = calculate_rsi(price_data[f'ssa_{window}'], period=period)
    
    # Add RSI for original (non-denoised) data as reference
    for period in rsi_periods:
        rsi_data[f'original_rsi_{period}'] = calculate_rsi(price_data['original'], period=period)
    
    # Drop NA values
    rsi_data = rsi_data.dropna()
    
    # Basic statistics of RSI data
    print("RSI DataFrame Shape:", rsi_data.shape)
    print("\nRSI Columns:")
    for col in rsi_data.columns:
        print(f"- {col}")
    
    print("\nRSI Basic Statistics:")
    print(rsi_data.describe().T[['mean', 'std', 'min', 'max']])
    
    # Example: Saving to CSV
    rsi_data.to_csv('btc_rsi_indicators.csv')

Progress: 1/3 chunks downloaded - 1000 data points so far
Progress: 2/3 chunks downloaded - 2000 data points so far
Progress: 3/3 chunks downloaded - 2556 data points so far
RSI DataFrame Shape: (2532, 56)

RSI Columns:
- ema_20_rsi_14
- ema_50_rsi_20
- ema_10_rsi_10
- ema_30_rsi_25
- wavelet_haar_rsi_10
- wavelet_haar_rsi_14
- wavelet_haar_rsi_20
- wavelet_haar_rsi_25
- wavelet_db2_rsi_10
- wavelet_db2_rsi_14
- wavelet_db2_rsi_20
- wavelet_db2_rsi_25
- wavelet_db4_rsi_10
- wavelet_db4_rsi_14
- wavelet_db4_rsi_20
- wavelet_db4_rsi_25
- wavelet_db6_rsi_10
- wavelet_db6_rsi_14
- wavelet_db6_rsi_20
- wavelet_db6_rsi_25
- wavelet_bior1.3_rsi_10
- wavelet_bior1.3_rsi_14
- wavelet_bior1.3_rsi_20
- wavelet_bior1.3_rsi_25
- wavelet_bior2.2_rsi_10
- wavelet_bior2.2_rsi_14
- wavelet_bior2.2_rsi_20
- wavelet_bior2.2_rsi_25
- ssa_5_rsi_10
- ssa_5_rsi_14
- ssa_5_rsi_20
- ssa_5_rsi_25
- ssa_10_rsi_10
- ssa_10_rsi_14
- ssa_10_rsi_20
- ssa_10_rsi_25
- ssa_20_rsi_10
- ssa_20_rsi_14
- ssa_20_rsi_20
- ss

### Load data and run DNN-HMM for every column.